# From Diversity to Prediction: Gut Microbiome Signatures of Colorectal Cancer and Ageing

The work focuses on how gut microbiome composition relates to colorectal cancer (CRC) and host age using diversity analysis, ecological statistics, classification models, and regression.

## Project Scope

This combined notebook organizes the assignment into four analytical themes:

1. Alpha diversity interpretation for CRC versus control.
2. Beta diversity, PERMANOVA testing, and PCoA visualization.
3. CRC classification using Random Forest and XGBoost.
4. Age prediction using feature-selected linear regression.

The underlying assignment used the Yachida et al. 2019 gut microbiome dataset. The raw dataset files are not included in this project folder, so the notebook is designed as a polished report with cleaned code blocks and consolidated interpretations.

## 1. Alpha Diversity Snapshot

The submitted assignment PDF reported alpha-diversity comparisons between CRC and control samples using Shannon diversity and Pielou's evenness, interpreted with a Wilcoxon rank-sum framework.

### Interpretation

- Shannon diversity appeared slightly higher in CRC samples than in controls.
- Pielou's evenness also showed a slight increase in CRC samples.
- In both cases the group distributions overlapped substantially, suggesting only modest separation between CRC and control at the alpha-diversity level.

### Takeaway

Alpha diversity alone did not provide strong separation between disease groups, which motivated the follow-up beta-diversity and machine-learning analyses.

## 2. Beta Diversity, PERMANOVA, and Ordination

The beta-diversity workflow used:

- Total-sum scaling (TSS) normalization on species abundances.
- Bray-Curtis dissimilarity to capture abundance-sensitive ecological distances.
- PERMANOVA to test whether microbiome composition differed by disease status and age.
- PCoA to visualize sample separation.

### Why Bray-Curtis?

Bray-Curtis was appropriate here because it:

- accounts for abundance rather than only presence/absence,
- handles sparse microbiome data well,
- ignores joint absences that are common in high-dimensional microbial tables.

### PERMANOVA Summary

| Model | F-value | R² | p-value |
| --- | ---: | ---: | ---: |
| `disease_condition` | 2.23 | 0.00443 | 0.005 |
| `age` | 4.98 | 0.00982 | 0.001 |
| `disease_condition + age` | 3.63 | 0.01427 | 0.001 |

### Interpretation

Disease condition and age both showed statistically significant associations with gut microbiome composition, but the low R² values indicate that the observed effects explain only a small part of the total variation.

![PCoA plot](pcoa_crc_vs_control.png)

### PCoA Interpretation

- `PC1 = 15.84%`
- `PC2 = 9.52%`
- The first two axes together explained about `25.36%` of the variance.
- CRC and control samples overlapped substantially, but a slight distributional shift was still visible.
- The overlap of the confidence ellipses supports the idea that the effect size is real but small.

### Cleaned R Workflow Used for Beta Diversity Analysis

```r
library(vegan)
library(ggplot2)

# Load species abundance and metadata tables.
species_profile <- read.csv("YachidaS_2019_SpeciesProfile.csv",
                            row.names = 1,
                            check.names = FALSE)
metadata <- read.csv("YachidaS_2019_Metadata.csv",
                     row.names = 1,
                     check.names = FALSE)

stopifnot(all(rownames(species_profile) == rownames(metadata)))
metadata$disease_condition <- as.factor(metadata$disease_condition)

# Total-sum scaling and Bray-Curtis beta diversity.
species_tss <- species_profile / rowSums(species_profile)
species_tss[is.na(species_tss)] <- 0
beta_dist_matrix <- vegdist(species_tss, method = "bray")

# PERMANOVA tests.
permanova_disease <- adonis2(beta_dist_matrix ~ disease_condition,
                             data = metadata,
                             permutations = 999)
permanova_age <- adonis2(beta_dist_matrix ~ age,
                         data = metadata,
                         permutations = 999)
permanova_combined <- adonis2(beta_dist_matrix ~ disease_condition + age,
                              data = metadata,
                              permutations = 999)

# PCoA visualization.
pcoa <- cmdscale(beta_dist_matrix, eig = TRUE, k = 2)
variance_explained <- round(pcoa$eig / sum(pcoa$eig) * 100, 2)

pcoa_df <- data.frame(
  PC1 = pcoa$points[, 1],
  PC2 = pcoa$points[, 2],
  disease_condition = metadata$disease_condition
)

pcoa_plot <- ggplot(pcoa_df, aes(x = PC1, y = PC2, color = disease_condition)) +
  geom_point(size = 3, alpha = 0.8) +
  stat_ellipse(level = 0.95, linetype = 2) +
  labs(
    title = "PCoA of Gut Microbiome Beta Diversity (CRC vs Control)",
    x = paste0("PC1 (", variance_explained[1], "%)"),
    y = paste0("PC2 (", variance_explained[2], "%)"),
    color = "Disease Condition"
  ) +
  theme_minimal()

print(permanova_disease)
print(permanova_age)
print(permanova_combined)
print(pcoa_plot)
```

## 3. CRC Classification with Random Forest and XGBoost

The classification workflow treated `disease_condition` as the response and species-level abundances as predictors.

### Preprocessing Strategy

- Remove taxa present in fewer than 10% of samples.
- Apply CLR transformation to reduce compositional bias.
- Split data into 75% training and 25% testing sets.
- Compare Random Forest and XGBoost.

### Model Performance

| Model | Accuracy | AUC |
| --- | ---: | ---: |
| Random Forest | 0.70 | 0.766 |
| XGBoost | 0.65-0.66 | 0.760 |

### Key Interpretation

- Random Forest slightly outperformed XGBoost in both accuracy and AUC.
- The microbiome carried a usable CRC signal, but not one strong enough for near-perfect classification.
- Random Forest was therefore used to rank the most informative taxa.

### Top Random Forest Taxa

1. `Gemella_morbillorum`
2. `Roseburia_faecis`
3. `Parvimonas_micra`
4. `Holdemania_filiformis`
5. `Peptostreptococcus_stomatis`

Based on the original interpretation, `Roseburia_faecis` was relatively enriched in controls, while the remaining leading taxa were relatively higher in CRC, consistent with a shift toward a more inflammation-associated gut environment.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from skbio.stats.composition import clr
from xgboost import XGBClassifier

species = pd.read_csv("YachidaS_2019_SpeciesProfile.csv", index_col=0)
metadata = pd.read_csv("YachidaS_2019_Metadata.csv", index_col=0)

# Remove sparse taxa present in fewer than 10% of samples.
prevalence = (species > 0).sum(axis=0)
threshold = 0.1 * species.shape[0]
filtered_species = species.loc[:, prevalence > threshold].copy()

filtered_species["condition"] = metadata["disease_condition"]
X_raw = filtered_species.drop(columns="condition")
y = filtered_species["condition"]
feature_names = X_raw.columns

# CLR transform to reduce compositional bias.
X = clr(X_raw.values + 1e-7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

rf = RandomForestClassifier(n_estimators=500, random_state=154)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print("Random Forest accuracy:", round(rf_accuracy, 3))
print(classification_report(y_test, rf_pred))

encoder = LabelEncoder()
y_train_enc = encoder.fit_transform(y_train)
y_test_enc = encoder.transform(y_test)

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
)
xgb.fit(X_train, y_train_enc)
xgb_pred = xgb.predict(X_test)
print("XGBoost accuracy:", round(accuracy_score(y_test_enc, xgb_pred), 3))
print(classification_report(y_test_enc, xgb_pred))

rf_probs = rf.predict_proba(X_test)[:, 1]
xgb_probs = xgb.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test_enc, rf_probs)
xgb_auc = roc_auc_score(y_test_enc, xgb_probs)

rf_fpr, rf_tpr, _ = roc_curve(y_test_enc, rf_probs, pos_label=1)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test_enc, xgb_probs, pos_label=1)

plt.plot(rf_fpr, rf_tpr, label=f"RF (AUC = {rf_auc:.2f})")
plt.plot(xgb_fpr, xgb_tpr, label=f"XGB (AUC = {xgb_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

feature_importance = (
    pd.DataFrame({"feature": feature_names, "importance": rf.feature_importances_})
    .sort_values(by="importance", ascending=False)
    .head(5)
)
print(feature_importance)

## 4. Age Prediction from Gut Microbiome Composition

The final analysis used linear regression to predict host age from normalized taxa abundances after feature selection.

### Feature Selection Logic

- Remove constant features.
- Remove highly sparse features with more than 95% zeros.
- Retain higher-variance taxa.
- Keep taxa with measurable correlation to age.
- Reduce multicollinearity by dropping highly correlated predictors.

### Results

- `MAE = 9.02 years`
- `Pearson r = 0.1966`
- `p = 0.027391`
- `Features retained = 87`

![Age prediction plot](age_prediction_regression.png)

### Interpretation

The feature-selected model performed much better than the original baseline, but the relationship between microbiome composition and chronological age remained weak. This suggests that the gut microbiome contains some age-related information, while diet, medication, disease status, lifestyle, and other confounders likely limit predictive power when age is modeled from abundance data alone.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

metadata = pd.read_csv("YachidaS_2019_Metadata.csv")
species = pd.read_csv("YachidaS_2019_SpeciesProfile.csv")

if "Unnamed: 0" in metadata.columns:
    metadata = metadata.drop(columns=["Unnamed: 0"])
species = species.rename(columns={species.columns[0]: "sample_id"})
data = metadata.merge(species, on="sample_id")

metadata_columns = ["sample_id", "age", "gender", "disease_condition"]
taxa_columns = [column for column in data.columns if column not in metadata_columns]
abundance_data = data[taxa_columns]

# TSS normalization.
abundance_normalized = abundance_data.div(abundance_data.sum(axis=1), axis=0).fillna(0)

# Feature selection.
abundance_normalized = abundance_normalized.loc[:, abundance_normalized.nunique() > 1]
retained_columns = [
    column
    for column in abundance_normalized.columns
    if (abundance_normalized[column] == 0).sum() / len(abundance_normalized) <= 0.95
]
abundance_normalized = abundance_normalized[retained_columns]

variance = abundance_normalized.var()
variance_threshold = variance.quantile(0.40)
X = abundance_normalized[variance[variance > variance_threshold].index]
y = data["age"]

correlations = X.corrwith(y).abs()
age_informative_taxa = correlations[correlations > 0.05].index
if len(age_informative_taxa) > 0:
    X = X[age_informative_taxa]
else:
    X = X[correlations.nlargest(int(len(correlations) * 0.5)).index]

correlation_matrix = X.corr().abs()
columns_to_drop = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[j, i] > 0.95:
            columns_to_drop.append(correlation_matrix.columns[i])
X = X.drop(columns=sorted(set(columns_to_drop)))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
correlation, p_value = pearsonr(y_test, y_pred)
print(f"MAE: {mae:.2f} years")
print(f"Pearson r: {correlation:.4f}")
print(f"P-value: {p_value:.6f}")
print("Features:", X.shape[1])

plt.figure(figsize=(9, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color="forestgreen")
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2,
    label="Perfect Prediction",
)
plt.xlabel("Actual Age (years)")
plt.ylabel("Predicted Age (years)")
plt.title("Actual vs Predicted Age - Linear Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Final Conclusion

From Diversity to Prediction: Gut Microbiome Signatures of Colorectal Cancer and Ageing shows that the gut microbiome carries:

- subtle but statistically significant differences between CRC and control groups,
- machine-learning signal that can support CRC classification,
- limited but detectable information related to host age.

Taken together, the project demonstrates an end-to-end microbiome analytics workflow that moves from ecological diversity analysis to predictive modeling. It also highlights an important biological lesson: significant microbiome effects are not always large effects, and predictive performance depends heavily on both feature engineering and the biological complexity of the phenotype being modeled.